In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import string

from sklearn.model_selection import train_test_split

## 1. Data Collection and Loading

The dataset contains sentiment-labelled sentences from three different domains:
- Amazon product reviews
- IMDb movie reviews
- Yelp restaurant reviews

Each review is labelled as positive (1) or negative (0).

In [ ]:
import os

PROJECT_PATH = 'PUT THE PATH HERE'
# Set the path to your project folder in Google Drive.
# Make sure Google Drive is mounted before running this cell.
# Replace the example path with the location where you saved the project.

DATASET_PATH = os.path.join(PROJECT_PATH, 'dataset')
MODEL_PATH = os.path.join(PROJECT_PATH, 'models')
NOTEBOOK_PATH = os.path.join(PROJECT_PATH, 'notebooks')

print(DATASET_PATH)

In [ ]:
os.listdir(DATASET_PATH)

In [ ]:
amazon = pd.read_csv(
    os.path.join(DATASET_PATH, 'amazon_cells_labelled.txt'),
    sep='\t',
    header=None,
    names=['text', 'sentiment']
)

imdb = pd.read_csv(
    os.path.join(DATASET_PATH, 'imdb_labelled.txt'),
    sep='\t',
    header=None,
    names=['text', 'sentiment']
)

yelp = pd.read_csv(
    os.path.join(DATASET_PATH, 'yelp_labelled.txt'),
    sep='\t',
    header=None,
    names=['text', 'sentiment']
)

In [ ]:
print("Amazon:", amazon.shape)
print("IMDb:", imdb.shape)
print("Yelp:", yelp.shape)

In [ ]:
amazon["source"] = "Amazon"
imdb["source"] = "IMDb"
yelp["source"] = "Yelp"

In [ ]:
df = pd.concat(
    [amazon, imdb, yelp],
    ignore_index=True
)

In [ ]:
print("Total records:", len(df))
print(df.shape)

In [ ]:
df["source"].value_counts()

In [ ]:
print(df["sentiment"].value_counts())

In [ ]:
print(
    pd.crosstab(
        df["source"],
        df["sentiment"]
    )
)

In [ ]:
pd.set_option("display.max_colwidth", 200)

df[["text", "sentiment", "source"]].sample(10, random_state=42)

In [ ]:
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
stop_words = set(stopwords.words('english'))

# Keep negation words because they are important for sentiment
negation_words = {
    'no', 'not', 'nor', 'never',
    "don't", "doesn't", "didn't",
    "isn't", "wasn't", "weren't",
    "can't", "couldn't",
    "won't", "wouldn't",
    "shouldn't"
}

stop_words = stop_words - negation_words

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean_text(text):
    # Convert to string
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Expand contractions while preserving the underscore
    contractions = {
        "isn't": "is_not",
        "wasn't": "was_not",
        "weren't": "were_not",
        "aren't": "are_not",
        "don't": "do_not",
        "doesn't": "does_not",
        "didn't": "did_not",
        "can't": "can_not",
        "couldn't": "could_not",
        "won't": "will_not",
        "wouldn't": "would_not",
        "shouldn't": "should_not",
        "haven't": "have_not",
        "hasn't": "has_not",
        "hadn't": "had_not",
        "it's": "it_is",
        "that's": "that_is",
        "there's": "there_is",
        "i'm": "i_am",
        "you're": "you_are",
        "we're": "we_are",
        "they're": "they_are",
        "i've": "i_have",
        "we've": "we_have",
        "they've": "they_have"
    }

    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Convert ratings such as 10/10 into a meaningful token
    text = re.sub(r'(\d+)\s*/\s*(\d+)', r'rating_\1_\2', text)

    # Remove non-ASCII / corrupted characters
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Remove punctuation EXCEPT underscore
    punctuation_without_underscore = string.punctuation.replace('_', '')
    text = text.translate(
        str.maketrans('', '', punctuation_without_underscore)
    )

    # Keep letters, underscores and spaces
    text = re.sub(r'[^a-zA-Z0-9_\s]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords and lemmatize
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return ' '.join(words)

In [ ]:
sample_text = "This movie was amazing! 10/10 would definitely recommend."

print("Original:")
print(sample_text)

print("\nCleaned:")
print(clean_text(sample_text))

In [ ]:
sample_text = "I ordered this product first and was unhappy with it immediately."

print("Original:")
print(sample_text)

print("\nCleaned:")
print(clean_text(sample_text))

In [ ]:
sample_text = "I do NOT like this product and I never recommend it."

print("Original:")
print(sample_text)

print("\nCleaned:")
print(clean_text(sample_text))

In [ ]:
sample_text = "I didn't like this product because it wasn't good."

print("Original:")
print(sample_text)

print("\nCleaned:")
print(clean_text(sample_text))

In [ ]:
df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
df[["text", "clean_text", "sentiment", "source"]].sample(
    10,
    random_state=42
)

In [ ]:
empty_reviews = df["clean_text"].str.strip().eq("").sum()

print("Empty cleaned reviews:", empty_reviews)

In [ ]:
empty_rows = df[df["clean_text"].str.strip().eq("")]

empty_rows[["text", "clean_text", "sentiment", "source"]]

In [ ]:
df["original_length"] = df["text"].str.split().str.len()
df["cleaned_length"] = df["clean_text"].str.split().str.len()

In [ ]:
print("Original text statistics:")
print(df["original_length"].describe())

print("\nCleaned text statistics:")
print(df["cleaned_length"].describe())

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="original_length",
    bins=30
)

plt.title("Distribution of Review Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Number of Reviews")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="sentiment",
    y="original_length"
)

plt.title("Review Length by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of Words")

plt.xticks(
    [0, 1],
    ["Negative", "Positive"]
)

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="source"
)

plt.title("Number of Reviews by Source")
plt.xlabel("Source")
plt.ylabel("Number of Reviews")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="source",
    hue="sentiment"
)

plt.title("Sentiment Distribution Across Sources")
plt.xlabel("Source")
plt.ylabel("Number of Reviews")

plt.legend(
    title="Sentiment",
    labels=["Negative", "Positive"]
)

plt.show()

In [ ]:
df.nlargest(10, "original_length")[[
    "text",
    "original_length",
    "sentiment",
    "source"
]]

In [ ]:
print(repr(imdb.iloc[136]["text"][:1000]))

In [ ]:
print("Number of embedded positive labels:",
      imdb["text"].str.count(r"\\t1\\n").sum())

print("Number of embedded negative labels:",
      imdb["text"].str.count(r"\\t0\\n").sum())

In [ ]:
print("Actual tab + label + newline:",
      imdb["text"].str.count("\t1\n").sum())

print("Actual tab + label + newline:",
      imdb["text"].str.count("\t0\n").sum())

In [ ]:
print(repr(imdb.iloc[136]["text"][:500]))

In [ ]:
imdb_file = os.path.join(
    DATASET_PATH,
    "imdb_labelled.txt"
)

print(imdb_file)

In [ ]:
imdb_records = []

with open(imdb_file, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        # Each valid record should end with a tab followed by 0 or 1
        if "\t0" in line or "\t1" in line:
            text, label = line.rsplit("\t", 1)

            if label in ["0", "1"]:
                imdb_records.append({
                    "text": text,
                    "sentiment": int(label),
                    "source": "IMDb"
                })

In [ ]:
imdb_fixed = pd.DataFrame(imdb_records)

print(imdb_fixed.shape)

In [ ]:
print(imdb_fixed["sentiment"].value_counts())

In [ ]:
print(imdb_fixed.head())

In [ ]:
# Make sure Amazon and Yelp have the source column
amazon["source"] = "Amazon"
yelp["source"] = "Yelp"

# Combine all three correctly parsed datasets
df = pd.concat(
    [amazon, imdb_fixed, yelp],
    ignore_index=True
)

print("Final dataset shape:", df.shape)

In [ ]:
print("\nSource distribution:")
print(df["source"].value_counts())

print("\nSentiment distribution:")
print(df["sentiment"].value_counts())

print("\nSentiment by source:")
print(pd.crosstab(df["source"], df["sentiment"]))

In [ ]:
print(df.head())
print(df.shape)

In [ ]:
print(df.isnull().sum())

In [ ]:
df["clean_text"] = df["text"].apply(clean_text)

In [ ]:
empty_reviews = df["clean_text"].str.strip().eq("").sum()

print("Empty cleaned reviews:", empty_reviews)

In [ ]:
df["original_length"] = df["text"].str.split().str.len()
df["cleaned_length"] = df["clean_text"].str.split().str.len()

print("Original text statistics:")
print(df["original_length"].describe())

print("\nCleaned text statistics:")
print(df["cleaned_length"].describe())

In [ ]:
df.nlargest(10, "original_length")[[
    "text",
    "original_length",
    "sentiment",
    "source"
]]

## 2. Exploratory Data Analysis

The final dataset contains 3,000 labelled text reviews from three domains:
Amazon, IMDb and Yelp.

Each source contributes 1,000 reviews, consisting of 500 positive and
500 negative reviews. The dataset therefore has a balanced sentiment
distribution.

The raw text and preprocessed text are analyzed using review-length
statistics and visualizations to understand the characteristics of
the textual data before feature extraction and model training.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="original_length",
    bins=30
)

plt.title("Distribution of Review Lengths")
plt.xlabel("Number of Words")
plt.ylabel("Number of Reviews")

plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="sentiment"
)

plt.title("Overall Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")

plt.xticks(
    [0, 1],
    ["Negative", "Positive"]
)

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(
    data=df,
    x="source"
)

plt.title("Number of Reviews by Source")
plt.xlabel("Source")
plt.ylabel("Number of Reviews")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="source",
    hue="sentiment"
)

plt.title("Sentiment Distribution Across Sources")
plt.xlabel("Source")
plt.ylabel("Number of Reviews")

plt.legend(
    title="Sentiment",
    labels=["Negative", "Positive"]
)

plt.show()

## 3. Word Frequency Analysis

Word frequency analysis is performed to identify commonly occurring
terms in the preprocessed review text. Separate analyses are performed
for positive and negative reviews to understand the vocabulary associated
with each sentiment.

In [ ]:
from collections import Counter

def get_word_frequencies(text_series, top_n=20):
    words = " ".join(text_series).split()
    return Counter(words).most_common(top_n)

In [ ]:
overall_words = get_word_frequencies(
    df["clean_text"],
    top_n=20
)

overall_words

In [ ]:
words, counts = zip(*overall_words)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=list(counts),
    y=list(words)
)

plt.title("Top 20 Most Frequent Words")
plt.xlabel("Frequency")
plt.ylabel("Word")

plt.show()

In [ ]:
positive_words = get_word_frequencies(
    df[df["sentiment"] == 1]["clean_text"],
    top_n=20
)

positive_words

In [ ]:
negative_words = get_word_frequencies(
    df[df["sentiment"] == 0]["clean_text"],
    top_n=20
)

negative_words

In [ ]:
words, counts = zip(*positive_words)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=list(counts),
    y=list(words)
)

plt.title("Top 20 Words in Positive Reviews")
plt.xlabel("Frequency")
plt.ylabel("Word")

plt.show()

In [ ]:
words, counts = zip(*negative_words)

plt.figure(figsize=(10, 6))

sns.barplot(
    x=list(counts),
    y=list(words)
)

plt.title("Top 20 Words in Negative Reviews")
plt.xlabel("Frequency")
plt.ylabel("Word")

plt.show()

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

In [ ]:
positive_text = " ".join(
    df[df["sentiment"] == 1]["clean_text"]
)

wordcloud_positive = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(positive_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud_positive, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud - Positive Reviews")
plt.show()

In [ ]:
negative_text = " ".join(
    df[df["sentiment"] == 0]["clean_text"]
)

wordcloud_negative = WordCloud(
    width=1000,
    height=500,
    background_color="white"
).generate(negative_text)

plt.figure(figsize=(12, 6))
plt.imshow(wordcloud_negative, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud - Negative Reviews")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
print("Training distribution:")
print(y_train.value_counts())

print("\nTesting distribution:")
print(y_test.value_counts())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 4. Sentiment Classification

TF-IDF converts the preprocessed text into numerical feature vectors.
Multiple supervised machine-learning algorithms are trained and
evaluated for binary sentiment classification.

The first model used is Multinomial Naive Bayes.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
y_pred_nb = nb_model.predict(X_test_tfidf)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
nb_accuracy = accuracy_score(
    y_test,
    y_pred_nb
)

print("Naive Bayes Accuracy:", nb_accuracy)

In [ ]:
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["Negative", "Positive"]
))

In [ ]:
cm_nb = confusion_matrix(
    y_test,
    y_pred_nb
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_nb,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.title("Naive Bayes Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.show()

In [ ]:
results = []

results.append({
    "Model": "Multinomial Naive Bayes",
    "Accuracy": nb_accuracy,
    "Precision": 0.84,
    "Recall": 0.83,
    "F1-Score": 0.83
})

pd.DataFrame(results)

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
lr_model.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
y_pred_lr = lr_model.predict(X_test_tfidf)

In [ ]:
lr_accuracy = accuracy_score(
    y_test,
    y_pred_lr
)

print("Logistic Regression Accuracy:", lr_accuracy)

In [ ]:
print(classification_report(
    y_test,
    y_pred_lr,
    target_names=["Negative", "Positive"]
))

In [ ]:
cm_lr = confusion_matrix(
    y_test,
    y_pred_lr
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.show()

## Support Vector Machine (SVM)

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
svm_model = LinearSVC(
    random_state=42,
    max_iter=5000
)

In [ ]:
svm_model.fit(
    X_train_tfidf,
    y_train
)

In [ ]:
y_pred_svm = svm_model.predict(X_test_tfidf)

In [ ]:
svm_accuracy = accuracy_score(
    y_test,
    y_pred_svm
)

print("SVM Accuracy:", svm_accuracy)

In [ ]:
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Negative", "Positive"]
))

In [ ]:
cm_svm = confusion_matrix(
    y_test,
    y_pred_svm
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_svm,
    annot=True,
    fmt="d",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.show()

In [ ]:
results = [
    {
        "Model": "Multinomial Naive Bayes",
        "Accuracy": nb_accuracy,
        "Precision": 0.84,
        "Recall": 0.83,
        "F1-Score": 0.83
    },
    {
        "Model": "Logistic Regression",
        "Accuracy": lr_accuracy,
        "Precision": 0.82,
        "Recall": 0.82,
        "F1-Score": 0.82
    },
    {
        "Model": "Linear SVM",
        "Accuracy": svm_accuracy,
        "Precision": 0.83,
        "Recall": 0.82,
        "F1-Score": 0.82
    }
]

results_df = pd.DataFrame(results)

results_df

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="Accuracy"
)

plt.ylim(0.75, 0.90)
plt.title("Model Accuracy Comparison")
plt.xlabel("Machine Learning Model")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = []

models = {
    "Multinomial Naive Bayes": y_pred_nb,
    "Logistic Regression": y_pred_lr,
    "Linear SVM": y_pred_svm
}

for name, predictions in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1-Score": f1_score(y_test, predictions)
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="Accuracy"
)

plt.ylim(0.75, 0.90)

plt.title("Model Accuracy Comparison")
plt.xlabel("Machine Learning Model")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
error_analysis = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred_nb
})

error_analysis = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
]

print("Misclassified reviews:", len(error_analysis))

In [ ]:
error_analysis.sample(
    15,
    random_state=42
)

In [ ]:
false_positives = error_analysis[
    (error_analysis["actual"] == 0) &
    (error_analysis["predicted"] == 1)
]

print("False Positives:", len(false_positives))

false_positives.sample(
    10,
    random_state=42
)

In [ ]:
false_negatives = error_analysis[
    (error_analysis["actual"] == 1) &
    (error_analysis["predicted"] == 0)
]

print("False Negatives:", len(false_negatives))

false_negatives.sample(
    10,
    random_state=42
)

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)

    vectorized = tfidf.transform([cleaned])

    prediction = nb_model.predict(vectorized)[0]
    probabilities = nb_model.predict_proba(vectorized)[0]

    confidence = max(probabilities) * 100

    if prediction == 1:
        sentiment = "Positive 😊"
    else:
        sentiment = "Negative 😞"

    return sentiment, confidence

In [ ]:
review = "I absolutely loved this product!"

sentiment, confidence = predict_sentiment(review)

print("Review:", review)
print("Sentiment:", sentiment)
print(f"Confidence: {confidence:.2f}%")

## User Interface

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def analyze_review(review):
    if not review or not review.strip():
        return "Please enter a review.", "—"

    cleaned = clean_text(review)

    if not cleaned.strip():
        return "Unable to analyze this review.", "—"

    vectorized = tfidf.transform([cleaned])

    prediction = nb_model.predict(vectorized)[0]
    probabilities = nb_model.predict_proba(vectorized)[0]

    probability = probabilities[prediction] * 100

    if prediction == 1:
        sentiment = "Positive 😊"
    else:
        sentiment = "Negative 😞"

    return sentiment, f"{probability:.2f}%"

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)

    vectorized = tfidf.transform([cleaned])

    prediction = nb_model.predict(vectorized)[0]
    probabilities = nb_model.predict_proba(vectorized)[0]

    probability = probabilities[prediction] * 100

    if prediction == 1:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, probability

In [ ]:
!pip install -q -U gradio

In [ ]:
!pip install -q -U langchain langchain-google-genai

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

In [ ]:
def analyze_with_gemini(review):
    prompt = f"""
You are a sentiment analysis assistant.

Analyze the following customer review.

Review:
"{review}"

Determine:
1. Overall sentiment: Positive, Negative, or Neutral
2. Main reason for the sentiment
3. Key positive aspects, if any
4. Key negative aspects, if any

Give a concise answer suitable for a sentiment analysis dashboard.
"""

    response = client.interactions.create(
        model="gemini-3.8-flash",
        input=prompt
    )

    return response.output_text

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

print("LangChain Gemini imported successfully!")

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

print("LangChain Gemini model created successfully!")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI sentiment analysis assistant.

Analyze the following customer review.

Review:
{review}

Determine:

1. Overall sentiment: Positive, Negative, or Neutral
2. Main reason for the sentiment
3. Key positive aspects
4. Key negative aspects

Keep the response concise and suitable for a sentiment analysis dashboard.
""")

print("Prompt template created successfully!")

In [ ]:
chain = prompt | llm

print("LangChain pipeline created successfully!")

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class SentimentAnalysis(BaseModel):
    sentiment: str = Field(
        description="Overall sentiment: Positive, Negative, or Neutral"
    )

    reason: str = Field(
        description="Main reason for the overall sentiment"
    )

    positive_aspects: List[str] = Field(
        description="Important positive aspects mentioned in the review"
    )

    negative_aspects: List[str] = Field(
        description="Important negative aspects mentioned in the review"
    )


print("Structured output schema created!")

In [ ]:
structured_llm = llm.with_structured_output(SentimentAnalysis)

print("Structured Gemini model created!")

In [ ]:
structured_chain = prompt | structured_llm

print("Structured LangChain pipeline created!")

In [ ]:
def combined_analysis(review):
    # -----------------------------
    # 1. Traditional ML Prediction
    # -----------------------------
    cleaned = clean_text(review)
    vectorized = tfidf.transform([cleaned])

    ml_prediction = nb_model.predict(vectorized)[0]
    probabilities = nb_model.predict_proba(vectorized)[0]
    ml_probability = probabilities[ml_prediction] * 100

    ml_sentiment = "Positive" if ml_prediction == 1 else "Negative"

    # -----------------------------
    # 2. LLM Analysis
    # -----------------------------
    ai_result = structured_chain.invoke({
        "review": review
    })

    ai_sentiment = ai_result.sentiment.strip().capitalize()

    # -----------------------------
    # 3. Compare ML and AI
    # -----------------------------
    disagreement = ml_sentiment != ai_sentiment

    # -----------------------------
    # 4. Prepare UI-friendly result
    # -----------------------------
    return {
        "ml_sentiment": ml_sentiment,
        "ml_probability": round(ml_probability, 2),
        "ai_sentiment": ai_sentiment,
        "ai_reason": ai_result.reason,
        "positive_aspects": ai_result.positive_aspects,
        "negative_aspects": ai_result.negative_aspects,
        "disagreement": disagreement
    }

In [ ]:
def clear_review():
    return ""

In [ ]:
# @title
import gradio as gr


# ============================================================
# SENTIMENT ANALYSIS
# ============================================================

def analyze_review(review):

    # ============================================================
    # EMPTY INPUT
    # ============================================================

    if not review or not review.strip():
        return """
        <div class="empty-state">
            <div class="empty-icon">✦</div>
            <div class="empty-title">Ready to analyze</div>
            <div class="empty-description">
                Enter a customer review above and click Analyze Sentiment.
            </div>
        </div>
        """

    # ============================================================
    # CLEANING CHECK
    # ============================================================

    cleaned = clean_text(review)

    if not cleaned.strip():
        return """
        <div class="empty-state warning-state">
            <div class="empty-icon">⚠️</div>
            <div class="empty-title">Unable to analyze</div>
            <div class="empty-description">
                Please enter meaningful text containing words or phrases.
            </div>
        </div>
        """

    # ============================================================
    # 1. TRADITIONAL ML MODEL
    # ============================================================

    cleaned_text = clean_text(review)
    vectorized = tfidf.transform([cleaned_text])

    ml_prediction = nb_model.predict(vectorized)[0]
    probabilities = nb_model.predict_proba(vectorized)[0]

    ml_probability = float(probabilities[ml_prediction] * 100)

    if ml_prediction == 1:
        ml_sentiment = "Positive"
    else:
        ml_sentiment = "Negative"

    # ============================================================
    # 2. LANGCHAIN + GEMINI AI ANALYSIS
    # ============================================================

    try:

        ai_result = structured_chain.invoke({
            "review": review
        })

        ai_sentiment = ai_result.sentiment.strip().capitalize()
        ai_reason = ai_result.reason

        positive_aspects = ai_result.positive_aspects
        negative_aspects = ai_result.negative_aspects

    except Exception as e:

        return f"""
        <div class="empty-state warning-state">
            <div class="empty-icon">⚠️</div>
            <div class="empty-title">AI analysis unavailable</div>
            <div class="empty-description">
                The machine learning model is available, but the AI
                interpretation could not be generated.
                <br><br>
                Error: {str(e)}
            </div>
        </div>
        """

    # ============================================================
    # 3. COMPARE ML + AI
    # ============================================================

    disagreement = ml_sentiment.lower() != ai_sentiment.lower()

    # ============================================================
    # 4. DETERMINE ML CARD STYLE
    # ============================================================

    if ml_sentiment == "Positive":

        ml_card_class = "positive-card"
        ml_text_class = "positive-text"
        ml_status_class = "positive-status"
        ml_fill_class = "positive-fill"
        ml_icon = "✓"
        ml_subtitle = "Positive language patterns detected"

    else:

        ml_card_class = "negative-card"
        ml_text_class = "negative-text"
        ml_status_class = "negative-status"
        ml_fill_class = "negative-fill"
        ml_icon = "!"
        ml_subtitle = "Negative language patterns detected"

    # ============================================================
    # 5. FORMAT ASPECTS
    # ============================================================

    positive_html = ""

    if positive_aspects:
        positive_items = "".join(
            f"<li>{aspect}</li>"
            for aspect in positive_aspects
        )

        positive_html = f"""
        <div class="aspect-group">
            <div class="aspect-title positive-aspect-title">
                ✓ Positive Aspects
            </div>

            <ul class="aspect-list">
                {positive_items}
            </ul>
        </div>
        """

    negative_html = ""

    if negative_aspects:
        negative_items = "".join(
            f"<li>{aspect}</li>"
            for aspect in negative_aspects
        )

        negative_html = f"""
        <div class="aspect-group">
            <div class="aspect-title negative-aspect-title">
                ! Negative Aspects
            </div>

            <ul class="aspect-list">
                {negative_items}
            </ul>
        </div>
        """

    # ============================================================
    # 6. DISAGREEMENT SECTION
    # ============================================================

    if disagreement:

        disagreement_html = f"""
        <div class="disagreement-card">

            <div class="disagreement-icon">
                ⚠️
            </div>

            <div class="disagreement-content">

                <div class="disagreement-title">
                    ML / AI Disagreement Detected
                </div>

                <div class="disagreement-text">
                    The statistical ML model classified this review as
                    <strong>{ml_sentiment}</strong>, while the AI
                    interpretation identified it as
                    <strong>{ai_sentiment}</strong>.
                </div>

                <div class="disagreement-note">
                    This indicates that contextual interpretation may
                    differ from the word-pattern-based ML prediction.
                </div>

            </div>

        </div>
        """

    else:

        disagreement_html = """
        <div class="agreement-card">

            <div class="agreement-icon">
                ✓
            </div>

            <div class="agreement-content">

                <div class="agreement-title">
                    ML / AI Agreement
                </div>

                <div class="agreement-text">
                    Both models identified the overall sentiment as
                    <strong>""" + ml_sentiment + """</strong>.
                </div>

            </div>

        </div>
        """

    # ============================================================
    # 7. FINAL RESULT
    # ============================================================

    return f"""

    <div class="hybrid-result-wrapper">

        <!-- ================================================ -->
        <!-- ML RESULT -->
        <!-- ================================================ -->

        <div class="result-card {ml_card_class}">

            <div class="result-header">

                <div>

                    <div class="result-eyebrow">
                        ML MODEL — MULTINOMIAL NAIVE BAYES
                    </div>

                    <div class="result-title {ml_text_class}">
                        {ml_sentiment}
                        <span>
                            {"😊" if ml_sentiment == "Positive" else "😞"}
                        </span>
                    </div>

                    <div class="result-subtitle">
                        {ml_subtitle}
                    </div>

                </div>

                <div class="result-status {ml_status_class}">
                    {ml_icon}
                </div>

            </div>

            <!-- Probability -->

            <div class="probability-area">

                <div class="probability-row">

                    <span>
                        Model Probability
                    </span>

                    <strong>
                        {ml_probability:.2f}%
                    </strong>

                </div>

                <div class="probability-track">

                    <div
                        class="probability-fill {ml_fill_class}"
                        style="width:{ml_probability}%"
                    ></div>

                </div>

                <div class="probability-note">
                    Estimated class probability from the trained
                    Multinomial Naive Bayes model.
                </div>

            </div>

        </div>


        <!-- ================================================ -->
        <!-- AI INTERPRETATION -->
        <!-- ================================================ -->

        <div class="ai-result-card">

            <div class="ai-header">

                <div>

                    <div class="result-eyebrow">
                        AI INTERPRETATION — LANGCHAIN + GEMINI
                    </div>

                    <div class="ai-title
                        {"ai-positive" if ai_sentiment == "Positive"
                         else "ai-negative" if ai_sentiment == "Negative"
                         else "ai-neutral"}">

                        {ai_sentiment}

                        <span>
                            {"😊" if ai_sentiment == "Positive"
                             else "😞" if ai_sentiment == "Negative"
                             else "😐"}
                        </span>

                    </div>

                </div>

                <div class="ai-badge">
                    🤖 AI
                </div>

            </div>


            <!-- AI Reason -->

            <div class="ai-reason">

                <div class="ai-section-label">
                    WHY?
                </div>

                <div class="ai-reason-text">
                    {ai_reason}
                </div>

            </div>


            <!-- AI Aspects -->

            <div class="aspect-grid">

                {positive_html}

                {negative_html}

            </div>

        </div>


        <!-- ================================================ -->
        <!-- COMPARISON -->
        <!-- ================================================ -->

        {disagreement_html}

    </div>

    """


# ============================================================
# EXAMPLE REVIEWS
# ============================================================

positive_example = (
    "I absolutely loved this product! "
    "The quality is excellent and I would definitely recommend it."
)

negative_example = (
    "This was terrible and a complete waste of money. "
    "The quality was extremely poor."
)

movie_example = (
    "The movie was amazing and I would definitely recommend it."
)

mixed_example = (
    "The product looks great but the quality is disappointing "
    "and it stopped working after two days."
)


# ============================================================
# CSS
# ============================================================

css = """

/* ============================================================
   GLOBAL
   ============================================================ */

body {
    background: #09090b !important;
}

.gradio-container {

    max-width: 1100px !important;

    margin: auto !important;

    padding: 30px 25px 70px !important;

    color: #f4f4f5 !important;
}

.gr-box,
.gr-panel,
.gr-form {
    border: none !important;
}


/* ============================================================
   HERO
   ============================================================ */

.hero {

    text-align: center;

    padding: 30px 10px 42px;
}

.hero-icon {

    width: 72px;
    height: 72px;

    margin: 0 auto 20px;

    display: flex;

    align-items: center;
    justify-content: center;

    border-radius: 22px;

    font-size: 34px;

    background:
        linear-gradient(
            135deg,
            #6366f1,
            #8b5cf6
        );

    box-shadow:
        0 15px 45px
        rgba(99,102,241,0.28);
}

.hero-title {

    font-size: 44px;

    line-height: 1.1;

    font-weight: 800;

    letter-spacing: -1.5px;

    background:
        linear-gradient(
            90deg,
            #ffffff,
            #c4b5fd
        );

    -webkit-background-clip: text;

    -webkit-text-fill-color: transparent;
}

.hero-subtitle {

    margin-top: 12px;

    color: #a1a1aa;

    font-size: 15px;
}

.online-pill {

    display: inline-flex;

    align-items: center;

    gap: 8px;

    margin-top: 18px;

    padding: 7px 13px;

    border-radius: 30px;

    background:
        rgba(34,197,94,0.08);

    border:
        1px solid
        rgba(34,197,94,0.20);

    color: #86efac;

    font-size: 11px;

    font-weight: 700;

    letter-spacing: 0.7px;
}

.online-dot {

    width: 7px;
    height: 7px;

    border-radius: 50%;

    background: #22c55e;

    box-shadow:
        0 0 10px
        rgba(34,197,94,0.8);
}


/* ============================================================
   INPUT AREA
   ============================================================ */

.input-area {

    background:
        linear-gradient(
            145deg,
            rgba(24,24,27,0.96),
            rgba(18,18,21,0.96)
        );

    border:
        1px solid #27272a;

    border-radius: 20px;

    padding: 28px;

    box-shadow:
        0 20px 60px
        rgba(0,0,0,0.25);
}

.section-heading {

    font-size: 19px;

    font-weight: 750;

    color: #fafafa;

    margin-bottom: 5px;
}

.section-description {

    color: #71717a;

    font-size: 13px;

    line-height: 1.5;

    margin-bottom: 20px;
}


/* ============================================================
   TEXTBOX
   ============================================================ */

.review-box textarea {

    background: #0d0d0f !important;

    border:
        1px solid #27272a !important;

    border-radius: 14px !important;

    color: #f4f4f5 !important;

    font-size: 15px !important;

    line-height: 1.65 !important;

    padding: 18px !important;

    min-height: 175px !important;

    transition: 0.2s ease !important;
}

.review-box textarea:focus {

    border-color:
        #6366f1 !important;

    box-shadow:
        0 0 0 2px
        rgba(99,102,241,0.12) !important;
}


/* ============================================================
   BUTTONS
   ============================================================ */

.analyze-btn {

    height: 52px !important;

    border-radius: 12px !important;

    border: none !important;

    background:
        linear-gradient(
            135deg,
            #6366f1,
            #8b5cf6
        ) !important;

    color: white !important;

    font-weight: 750 !important;

    font-size: 15px !important;

    transition: 0.2s ease !important;
}

.analyze-btn:hover {

    transform:
        translateY(-2px);

    box-shadow:
        0 12px 30px
        rgba(99,102,241,0.28);
}

.clear-btn {

    height: 52px !important;

    border-radius: 12px !important;

    border:
        1px solid #27272a !important;

    background: #18181b !important;

    color: #a1a1aa !important;

    font-weight: 650 !important;

    transition: 0.2s ease !important;
}

.clear-btn:hover {

    background: #27272a !important;

    color: #f4f4f5 !important;
}


/* ============================================================
   RESULT
   ============================================================ */

.result-heading {

    margin-top: 34px;

    margin-bottom: 14px;
}

.result-card {

    padding: 27px;

    border-radius: 18px;

    position: relative;

    overflow: hidden;
}

.positive-card {

    background:
        linear-gradient(
            135deg,
            rgba(22,101,52,0.17),
            rgba(18,18,21,0.96)
        );

    border:
        1px solid
        rgba(34,197,94,0.25);
}

.negative-card {

    background:
        linear-gradient(
            135deg,
            rgba(127,29,29,0.17),
            rgba(18,18,21,0.96)
        );

    border:
        1px solid
        rgba(239,68,68,0.25);
}

.result-header {

    display: flex;

    align-items: center;

    justify-content: space-between;
}

.result-eyebrow {

    color: #71717a;

    font-size: 10px;

    letter-spacing: 1.7px;

    font-weight: 750;
}

.result-title {

    font-size: 34px;

    font-weight: 850;

    letter-spacing: -0.8px;

    margin-top: 4px;
}

.positive-text {
    color: #4ade80;
}

.negative-text {
    color: #f87171;
}

.result-title span {
    font-size: 27px;
}

.result-subtitle {

    color: #a1a1aa;

    font-size: 12px;

    margin-top: 4px;
}

.result-status {

    width: 58px;
    height: 58px;

    border-radius: 17px;

    display: flex;

    align-items: center;

    justify-content: center;

    font-size: 28px;

    font-weight: 800;
}

.positive-status {

    background:
        rgba(34,197,94,0.13);

    color: #4ade80;
}

.negative-status {

    background:
        rgba(239,68,68,0.13);

    color: #f87171;
}


/* ============================================================
   PROBABILITY
   ============================================================ */

.probability-area {

    margin-top: 28px;
}

.probability-row {

    display: flex;

    align-items: center;

    justify-content: space-between;

    color: #a1a1aa;

    font-size: 12px;

    margin-bottom: 9px;
}

.probability-row strong {

    color: #f4f4f5;

    font-size: 15px;
}

.probability-track {

    width: 100%;

    height: 8px;

    background: #27272a;

    border-radius: 20px;

    overflow: hidden;
}

.probability-fill {

    height: 100%;

    border-radius: 20px;
}

.positive-fill {

    background:
        linear-gradient(
            90deg,
            #22c55e,
            #86efac
        );
}

.negative-fill {

    background:
        linear-gradient(
            90deg,
            #ef4444,
            #fb7185
        );
}

.probability-note {

    margin-top: 7px;

    color: #52525b;

    font-size: 10px;
}

.result-note {

    margin-top: 18px;

    color: #71717a;

    font-size: 12px;

    line-height: 1.6;
}


/* ============================================================
   EMPTY STATE
   ============================================================ */

.empty-state {

    padding: 48px 25px;

    text-align: center;

    border:
        1px dashed #3f3f46;

    border-radius: 18px;

    background: #111113;
}

.empty-icon {

    font-size: 32px;

    color: #8b5cf6;
}

.empty-title {

    color: #e4e4e7;

    font-size: 17px;

    font-weight: 700;

    margin-top: 10px;
}

.empty-description {

    color: #71717a;

    font-size: 12px;

    margin-top: 6px;
}


/* ============================================================
   STATS
   ============================================================ */

.stats-grid {

    display: grid;

    grid-template-columns:
        repeat(4, 1fr);

    gap: 12px;

    margin-top: 15px;
}

.stat-card {

    padding: 18px;

    background: #111113;

    border:
        1px solid #27272a;

    border-radius: 14px;
}

.stat-label {

    color: #52525b;

    font-size: 9px;

    letter-spacing: 1.2px;

    font-weight: 700;
}

.stat-value {

    color: #e4e4e7;

    font-size: 18px;

    font-weight: 750;

    margin-top: 7px;
}

.stat-accent {
    color: #a78bfa;
}


/* ============================================================
   EXAMPLE CARDS
   ============================================================ */

.examples-section {

    margin-top: 35px;
}

.example-grid {

    display: grid;

    grid-template-columns:
        repeat(2, 1fr);

    gap: 10px;

    margin-top: 15px;
}

.example-btn {

    text-align: left !important;

    min-height: 105px !important;

    padding: 16px !important;

    border-radius: 14px !important;

    background: #111113 !important;

    border:
        1px solid #27272a !important;

    color: #f4f4f5 !important;

    transition: 0.2s ease !important;
}

.example-btn:hover {

    border-color:
        #6366f1 !important;

    background:
        #15151a !important;

    transform:
        translateY(-2px);
}

.example-label {

    display: block;

    font-size: 12px;

    font-weight: 750;

    margin-bottom: 8px;
}

.example-text {

    display: block;

    color: #71717a;

    font-size: 11px;

    line-height: 1.5;

    white-space: normal;
}


/* ============================================================
   INFO CARDS
   ============================================================ */

.info-card {

    margin-top: 30px;

    padding: 25px;

    background:
        linear-gradient(
            135deg,
            rgba(99,102,241,0.07),
            rgba(139,92,246,0.025)
        );

    border:
        1px solid
        rgba(99,102,241,0.16);

    border-radius: 18px;
}


/* ============================================================
   PIPELINE
   ============================================================ */

.pipeline {

    display: flex;

    justify-content: center;

    align-items: center;

    flex-wrap: wrap;

    gap: 9px;

    margin-top: 20px;
}

.pipeline-node {

    padding: 10px 14px;

    border-radius: 10px;

    background: #18181b;

    border:
        1px solid #27272a;

    color: #d4d4d8;

    font-size: 11px;

    font-weight: 600;
}

.pipeline-arrow {

    color: #6366f1;

    font-weight: 800;

    font-size: 14px;
}


/* ============================================================
   MODEL COMPARISON
   ============================================================ */

.model-comparison {
    margin-top: 18px;
}

.model-row {
    margin-bottom: 16px;
}

.model-row-header {

    display: flex;

    justify-content: space-between;

    color: #a1a1aa;

    font-size: 11px;

    margin-bottom: 6px;
}

.model-row-header strong {
    color: #e4e4e7;
}

.model-track {

    height: 7px;

    background: #27272a;

    border-radius: 20px;

    overflow: hidden;
}

.model-fill {

    height: 100%;

    border-radius: 20px;

    background:
        linear-gradient(
            90deg,
            #6366f1,
            #8b5cf6
        );
}

.best-model .model-fill {

    background:
        linear-gradient(
            90deg,
            #22c55e,
            #86efac
        );
}


/* ============================================================
   FOOTER
   ============================================================ */

.footer {

    text-align: center;

    color: #52525b;

    font-size: 11px;

    line-height: 1.8;

    margin-top: 40px;
}


/* ============================================================
   MOBILE
   ============================================================ */

@media (max-width: 700px) {

    .gradio-container {
        padding:
            20px 12px 50px !important;
    }

    .hero-title {
        font-size: 31px;
    }

    .input-area {
        padding: 20px;
    }

    .stats-grid {
        grid-template-columns:
            repeat(2, 1fr);
    }

    .example-grid {
        grid-template-columns: 1fr;
    }

    .pipeline {
        flex-direction: column;
    }

    .pipeline-arrow {
        transform: rotate(90deg);
    }

    .result-title {
        font-size: 27px;
    }
}

/* ============================================================
   HYBRID AI RESULT
   ============================================================ */

.hybrid-result-wrapper {
    display: flex;
    flex-direction: column;
    gap: 14px;
}


/* ============================================================
   AI CARD
   ============================================================ */

.ai-result-card {

    padding: 27px;

    border-radius: 18px;

    position: relative;

    overflow: hidden;

    background:
        linear-gradient(
            135deg,
            rgba(99,102,241,0.10),
            rgba(18,18,21,0.97)
        );

    border:
        1px solid
        rgba(139,92,246,0.25);

}


.ai-header {

    display: flex;

    align-items: center;

    justify-content: space-between;

}


.ai-title {

    font-size: 30px;

    font-weight: 850;

    letter-spacing: -0.7px;

    margin-top: 5px;

}


.ai-positive {
    color: #4ade80;
}


.ai-negative {
    color: #f87171;
}


.ai-neutral {
    color: #facc15;
}


.ai-title span {

    font-size: 25px;

}


.ai-badge {

    padding: 8px 12px;

    border-radius: 10px;

    background:
        rgba(139,92,246,0.12);

    border:
        1px solid
        rgba(139,92,246,0.25);

    color: #c4b5fd;

    font-size: 11px;

    font-weight: 750;

}


/* ============================================================
   AI REASON
   ============================================================ */

.ai-reason {

    margin-top: 25px;

    padding: 17px;

    border-radius: 13px;

    background: #111113;

    border:
        1px solid #27272a;

}


.ai-section-label {

    color: #8b5cf6;

    font-size: 9px;

    font-weight: 800;

    letter-spacing: 1.4px;

    margin-bottom: 7px;

}


.ai-reason-text {

    color: #d4d4d8;

    font-size: 13px;

    line-height: 1.6;

}


/* ============================================================
   ASPECTS
   ============================================================ */

.aspect-grid {

    display: grid;

    grid-template-columns:
        repeat(2, 1fr);

    gap: 12px;

    margin-top: 12px;

}


.aspect-group {

    padding: 15px;

    border-radius: 13px;

    background: #111113;

    border:
        1px solid #27272a;

}


.aspect-title {

    font-size: 11px;

    font-weight: 750;

    margin-bottom: 9px;

}


.positive-aspect-title {

    color: #4ade80;

}


.negative-aspect-title {

    color: #f87171;

}


.aspect-list {

    margin: 0;

    padding-left: 18px;

    color: #a1a1aa;

    font-size: 11px;

    line-height: 1.7;

}


/* ============================================================
   DISAGREEMENT
   ============================================================ */

.disagreement-card {

    display: flex;

    align-items: flex-start;

    gap: 14px;

    padding: 18px;

    border-radius: 15px;

    background:
        rgba(245,158,11,0.07);

    border:
        1px solid
        rgba(245,158,11,0.25);

}


.disagreement-icon {

    width: 38px;

    height: 38px;

    min-width: 38px;

    border-radius: 11px;

    display: flex;

    align-items: center;

    justify-content: center;

    background:
        rgba(245,158,11,0.12);

    font-size: 18px;

}


.disagreement-title {

    color: #fbbf24;

    font-size: 13px;

    font-weight: 800;

}


.disagreement-text {

    color: #a1a1aa;

    font-size: 11px;

    line-height: 1.6;

    margin-top: 4px;

}


.disagreement-note {

    color: #71717a;

    font-size: 10px;

    line-height: 1.5;

    margin-top: 6px;

}


/* ============================================================
   AGREEMENT
   ============================================================ */

.agreement-card {

    display: flex;

    align-items: center;

    gap: 14px;

    padding: 17px;

    border-radius: 15px;

    background:
        rgba(34,197,94,0.06);

    border:
        1px solid
        rgba(34,197,94,0.18);

}


.agreement-icon {

    width: 38px;

    height: 38px;

    min-width: 38px;

    border-radius: 11px;

    display: flex;

    align-items: center;

    justify-content: center;

    background:
        rgba(34,197,94,0.10);

    color: #4ade80;

    font-weight: 800;

}


.agreement-title {

    color: #4ade80;

    font-size: 13px;

    font-weight: 800;

}


.agreement-text {

    color: #a1a1aa;

    font-size: 11px;

    margin-top: 4px;

}


/* ============================================================
   MOBILE — HYBRID RESULT
   ============================================================ */

@media (max-width: 700px) {

    .aspect-grid {

        grid-template-columns: 1fr;

    }

    .ai-title {

        font-size: 27px;

    }

}

"""


# ============================================================
# BUILD APP
# ============================================================

with gr.Blocks(
    title="Feedback Intelligence System"
) as demo:


    # ========================================================
    # HERO
    # ========================================================

    gr.HTML("""
    <div class="hero">

        <div class="hero-icon">
            🧠
        </div>

        <div class="hero-title">
            Feedback Intelligence System
        </div>

        <div class="hero-subtitle">
            AI-powered sentiment analysis for customer feedback
        </div>

        <div class="online-pill">
            <span class="online-dot"></span>
            MULTINOMIAL NAIVE BAYES ONLINE
        </div>

    </div>
    """)


    # ========================================================
    # INPUT
    # ========================================================

    with gr.Column(elem_classes="input-area"):

        gr.HTML("""
        <div class="section-heading">
            Analyze Customer Feedback
        </div>

        <div class="section-description">
            Enter a product, movie, restaurant, or service review
            and let the trained machine learning model analyze its sentiment.
        </div>
        """)

        review_input = gr.Textbox(
            show_label=False,

            placeholder=(
                "Example: The product quality is excellent "
                "and I would definitely recommend it..."
            ),

            lines=7,

            max_lines=10,

            elem_classes="review-box"
        )

        with gr.Row():

            clear_button = gr.Button(
                "🧹  Clear",
                elem_classes="clear-btn",
                scale=1
            )

            analyze_button = gr.Button(
                "🔍  Analyze Sentiment",
                elem_classes="analyze-btn",
                scale=2
            )


    # ========================================================
    # RESULT
    # ========================================================

    gr.HTML("""
    <div class="result-heading">

        <div class="section-heading">
            Sentiment Result
        </div>

        <div class="section-description">
            ML classification with contextual AI interpretation.
        </div>

    </div>
    """)

    result_output = gr.HTML("""
    <div class="empty-state">

        <div class="empty-icon">
            ✦
        </div>

        <div class="empty-title">
            Ready to analyze
        </div>

        <div class="empty-description">
            Enter a customer review and click Analyze Sentiment.
        </div>

    </div>
    """)


    # ========================================================
    # STATS
    # ========================================================

    gr.HTML("""
    <div class="stats-grid">

        <div class="stat-card">
            <div class="stat-label">
                TOTAL REVIEWS
            </div>
            <div class="stat-value">
                3,000
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">
                TRAINING
            </div>
            <div class="stat-value">
                2,400
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">
                TESTING
            </div>
            <div class="stat-value">
                600
            </div>
        </div>

        <div class="stat-card">
            <div class="stat-label">
                ACCURACY
            </div>
            <div class="stat-value stat-accent">
                83.50%
            </div>
        </div>

    </div>
    """)


    # ========================================================
    # EXAMPLES
    # ========================================================

    gr.HTML("""
    <div class="examples-section">

        <div class="section-heading">
            💡 Try an Example
        </div>

        <div class="section-description">
            Click an example to load it into the review box.
        </div>

    </div>
    """)


    with gr.Row(elem_classes="example-grid"):

        positive_btn = gr.Button(
            value="😊  POSITIVE\nI absolutely loved this product! The quality is excellent...",
            elem_classes="example-btn"
        )

        negative_btn = gr.Button(
            value="😞  NEGATIVE\nThis was terrible and a complete waste of money...",
            elem_classes="example-btn"
        )


    with gr.Row(elem_classes="example-grid"):

        movie_btn = gr.Button(
            value="🎬  MOVIE\nThe movie was amazing and I would definitely recommend it.",
            elem_classes="example-btn"
        )

        mixed_btn = gr.Button(
            value="⚠️  MIXED SENTIMENT\nThe product looks great but the quality is disappointing...",
            elem_classes="example-btn"
        )


    # ========================================================
    # HOW IT WORKS
    # ========================================================

    gr.HTML("""
    <div class="info-card">

        <div class="section-heading">
            ⚙️ How It Works
        </div>

        <div class="section-description">
            The system transforms raw customer feedback into
            machine-readable features before classification.
        </div>

        <div class="pipeline">

            <div class="pipeline-node">
                📝 Review
            </div>

            <div class="pipeline-arrow">→</div>

            <div class="pipeline-node">
                🧹 Cleaning
            </div>

            <div class="pipeline-arrow">→</div>

            <div class="pipeline-node">
                📊 TF-IDF
            </div>

            <div class="pipeline-arrow">→</div>

            <div class="pipeline-node">
                🧠 Multinomial NB
            </div>

            <div class="pipeline-arrow">→</div>

            <div class="pipeline-node">
                💡 Sentiment
            </div>

        </div>

    </div>
    """)


    # ========================================================
    # MODEL PERFORMANCE
    # ========================================================

    gr.HTML("""
    <div class="info-card">

        <div class="section-heading">
            📊 Model Performance
        </div>

        <div class="section-description">
            Performance comparison on the 600-review held-out test dataset.
        </div>

        <div class="model-comparison">

            <div class="model-row best-model">

                <div class="model-row-header">

                    <span>
                        Multinomial Naive Bayes
                    </span>

                    <strong>
                        83.50%
                    </strong>

                </div>

                <div class="model-track">

                    <div
                        class="model-fill"
                        style="width:83.5%"
                    ></div>

                </div>

            </div>


            <div class="model-row">

                <div class="model-row-header">

                    <span>
                        Linear SVM
                    </span>

                    <strong>
                        82.50%
                    </strong>

                </div>

                <div class="model-track">

                    <div
                        class="model-fill"
                        style="width:82.5%"
                    ></div>

                </div>

            </div>


            <div class="model-row">

                <div class="model-row-header">

                    <span>
                        Logistic Regression
                    </span>

                    <strong>
                        82.33%
                    </strong>

                </div>

                <div class="model-track">

                    <div
                        class="model-fill"
                        style="width:82.33%"
                    ></div>

                </div>

            </div>

        </div>


        <div class="section-description"
             style="margin-top:20px;margin-bottom:0">

            <strong style="color:#a78bfa">
                Best model:
            </strong>

            Multinomial Naive Bayes —
            <strong style="color:#4ade80">
                83.50%
            </strong>
            test accuracy.

        </div>

    </div>
    """)


    # ========================================================
    # DATASET
    # ========================================================

    gr.HTML("""
    <div class="info-card">

        <div class="section-heading">
            🗂️ Dataset Overview
        </div>

        <div class="section-description">
            Balanced review data collected from three different sources.
        </div>

        <div class="stats-grid">

            <div class="stat-card">

                <div class="stat-label">
                    AMAZON
                </div>

                <div class="stat-value">
                    1,000
                </div>

            </div>


            <div class="stat-card">

                <div class="stat-label">
                    IMDB
                </div>

                <div class="stat-value">
                    1,000
                </div>

            </div>


            <div class="stat-card">

                <div class="stat-label">
                    YELP
                </div>

                <div class="stat-value">
                    1,000
                </div>

            </div>


            <div class="stat-card">

                <div class="stat-label">
                    CLASSES
                </div>

                <div class="stat-value">
                    2
                </div>

            </div>

        </div>

    </div>
    """)


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML("""
    <div class="footer">

        <div>
            Feedback Intelligence System
            • AI-DS II
            • Sentiment Analysis
        </div>

        <div>
            TF-IDF
            • Multinomial Naive Bayes
            • 3,000 Reviews
        </div>

    </div>
    """)


    # ========================================================
    # EVENTS
    # ========================================================

    analyze_button.click(
        fn=analyze_review,
        inputs=review_input,
        outputs=result_output
    )

    review_input.submit(
        fn=analyze_review,
        inputs=review_input,
        outputs=result_output
    )

    clear_button.click(
        fn=clear_review,
        inputs=[],
        outputs=review_input
    )

    # Example buttons

    positive_btn.click(
        fn=lambda: positive_example,
        inputs=[],
        outputs=review_input
    )

    negative_btn.click(
        fn=lambda: negative_example,
        inputs=[],
        outputs=review_input
    )

    movie_btn.click(
        fn=lambda: movie_example,
        inputs=[],
        outputs=review_input
    )

    mixed_btn.click(
        fn=lambda: mixed_example,
        inputs=[],
        outputs=review_input
    )


# ============================================================
# LAUNCH
# ============================================================
# Launch the Gradio application.
# share=False keeps the app local and does not create a public Gradio share link.
# Set share=True only if you intentionally want to create a temporary public demo link.
demo.launch(
    share=False,
    css=css
    )